# PDA Distillation: Teaching Multi-Perspective Reasoning to a Single Model

**Goal:** Fine-tune Qwen3-1.7B to internalize the multi-perspective reasoning pattern
that PDA (Parallel Deliberation Architecture) uses with 3 workers + merger.

**Data:** 474 GSM8K training examples where PDA produced correct answers.
Each example has the merged reasoning from 3 workers (methodical, creative, skeptical).

**Hypothesis:** A single model can learn to reason from multiple perspectives
in one forward pass, matching PDA's accuracy without 4x compute cost.

**Hardware:** Colab Free T4 (16GB VRAM) — sufficient for QLoRA on 1.7B model.

## 1. Setup

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable params: {model.print_trainable_parameters()}")

## 2. Load Training Data

Upload `pda_training_data.jsonl` to Colab first (Files panel on the left).

In [ ]:
import json
from datasets import Dataset

# Load and filter to correct examples only
examples = []
with open("pda_training_data.jsonl") as f:
    for line in f:
        d = json.loads(line)
        if d["correct"]:
            examples.append(d)

print(f"Loaded {len(examples)} correct training examples")

# System prompt that teaches the multi-perspective pattern
SYSTEM_PROMPT = """You are a math problem solver who considers multiple approaches before answering. For each problem:
1. First, solve it step by step methodically.
2. Then, look for a more efficient approach or shortcut.
3. Finally, check for edge cases and common mistakes.
4. Synthesize the best answer from your analysis.

End with #### <number>"""

# Format as chat conversations
def format_example(ex):
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["question"]},
            {"role": "assistant", "content": ex["pda_reasoning"]},
        ]
    }

formatted = [format_example(ex) for ex in examples]
dataset = Dataset.from_list(formatted)
print(f"Dataset: {len(dataset)} examples")
print(f"Example conversation:\n{formatted[0]['conversations'][2]['content'][:300]}...")

## 3. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False
    ) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="pda-distilled",
    ),
)

trainer_stats = trainer.train()
print(f"Training loss: {trainer_stats.training_loss:.4f}")

## 4. Evaluation on GSM8K Test Set

Compare: Base Qwen3-1.7B vs PDA-Distilled Qwen3-1.7B

In [ ]:
import re
from datasets import load_dataset

# Load test set (same 200 questions as Sim 4 for comparability)
import random
random.seed(42)
test_ds = load_dataset("openai/gsm8k", "main", split="test")
indices = list(range(len(test_ds)))
random.shuffle(indices)
test_indices = indices[:200]

def extract_answer(text):
    match = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    if match:
        return float(match.group(1).replace(",", ""))
    numbers = re.findall(r'-?[\d,]+\.?\d*', text)
    for n in reversed(numbers):
        cleaned = n.replace(",", "").strip()
        if cleaned and cleaned != "-":
            try: return float(cleaned)
            except: continue
    return None

def extract_gsm8k_answer(text):
    match = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return float(match.group(1).replace(",", ""))

# Switch to inference mode
FastLanguageModel.for_inference(model)

correct = 0
total = 0
errors = []

for i, idx in enumerate(test_indices):
    item = test_ds[idx]
    gt = extract_gsm8k_answer(item["answer"])
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["question"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=512,
            temperature=0.7, do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    pred = extract_answer(response)
    
    if pred is not None and pred == gt:
        correct += 1
    else:
        errors.append({"q": item["question"][:60], "gt": gt, "pred": pred})
    total += 1
    
    if (i + 1) % 20 == 0:
        print(f"[{i+1}/200] Accuracy: {correct}/{total} ({100*correct/total:.1f}%)")

print(f"\n=== FINAL ===")
print(f"PDA-Distilled Qwen3-1.7B: {correct}/{total} ({100*correct/total:.1f}%)")
print(f"\nComparison (Qwen3-8B, Sim 4):")
print(f"  Baseline 8B:   192/200 (96.0%)")
print(f"  Prompt-PDA 8B: 195/200 (97.5%)")
print(f"  Distilled 1.7B: {correct}/{total} ({100*correct/total:.1f}%)")

## 5. Save Model

In [ ]:
# Save LoRA adapter
model.save_pretrained("pda-distilled-lora")
tokenizer.save_pretrained("pda-distilled-lora")
print("Saved to pda-distilled-lora/")

# Optional: merge and save full model (for Ollama etc.)
# model.save_pretrained_merged("pda-distilled-merged", tokenizer, save_method="merged_16bit")

# Optional: export to GGUF for local inference
# model.save_pretrained_gguf("pda-distilled-gguf", tokenizer, quantization_method="q4_k_m")

## Results

| Model | GSM8K (200q) | Compute |
|-------|-------------|--------|
| Qwen3-8B Baseline | 96.0% | 1x |
| Qwen3-8B Prompt-PDA | 97.5% | 4x |
| Qwen3-1.7B PDA-Distilled | ???% | 1x (5x smaller model) |

**Key question:** Does the distilled 1.7B model match or approach the 8B baseline,
despite being 5x smaller? If yes, PDA distillation compresses multi-perspective
reasoning into a single forward pass.